# Understanding Large Language Models (LLMs)

## What is an LLM?

An **Large Language Model (LLM)** is a type of artificial intelligence trained on vast amounts of text data to understand and generate human language. LLMs don't "understand" in the human sense, but rather they are trained to match patterns of statistical relationships in language. Because those patterns were written by humans (at least before generative AI existed), they encode information about the world that can be extracted by repeatedly extending the text by selecting the next **token** (or word chunk) from the most probable ones. The caveat is that generation is probablistic and the model outputs can therefor be factually incorrect (otherwise known as **hallucinations**).

## How LLMs Work

LLMs use **transformers**, a neural network architecture that processes text by:
1. Breaking text into tokens (words/subwords)
2. Processing all tokens in parallel with "attention" mechanisms
3. Computing probabilities for what token comes next
4. Sampling from those probabilities to generate output

LLMs (and neural networks in general) are composed of a structured set of **weights**, which are analagous to dials that adjust the probability distribution of the next token based on the surrounding **context**. At the start of an interaction, the LLM is extending off of any system-level instructions (the **system prompt**) that were provided to it plus the user input (**user prompt**). When a token is added to the text, the context is updated, probabilities recalculated, and next token predicted. This process is repeated until a special stop token is encountered.

It is important to note that everything that is passed to the model is essentially converted into text (tokens) and added to the current interaction. This feature of LLMs creates an inherent security vulnerability known as **prompt injection**, where malicious instructions are added to the context with the intention of influencing the model outputs. While model developers have been working to implement guardrails to prevent unintended behaviors, there is no way to completely prevent models from following injected instructions. **To be clear: if you are allowing a model to access outside resources, there is a risk of prompt injection.** Most popular applications (Claude Code, Codex, etc.) allow you to set permissions and review actions before they are performed.

For an excellent in-depth explainer, check out the LLM series from [3blue1brown](https://www.youtube.com/@3blue1brown) on YouTube.

### Tokens
Language models (including but not limited to LLMs) don't actually process text. Instead, the tokens are converted to numbers based on a vocabulary list and added to a vector. Tokenizers exist for converting whole or partial words to tokens, but LLMs specifically use sub-word tokenization (roughly ~4 characters per token). The advantages of this approach are:
- Smaller vocabulary list (and smaller memory footprints for the values)
    For example, take the comparative forms of "fast" and "slow": `[fast, faster, fastest]`, `[slow, slower, slowest]`. If tokenizing by whole words, all six words would need to be in the vocabulary.
    If you instead use sub-words, you could define four tokens from which you could compose all of the words: `[fast, slow, er, est]`.
- Create non-word outputs (code, symbols, abbreviations, etc)

The drawback is that longer/more complex words are split into multiple tokens, which not only splits the semantic information into multiple predictions but also increases the length of the input vector.
Longer inputs can cause issues:
- LLMs have a limit to the number of tokens that can fit in context
- Model providers charge per token (or limit the number of tokens for a time period in subscription models)

## Prompts and Roles

Communication with LLMs uses a **message-based format** where each message has a role indicating who's speaking:

### System Role
The **system message** sets the context and instructions for how the LLM should behave. It's separate from the conversation and shapes all responses.
```
System: You are a helpful coding assistant. Be concise and suggest best practices.
```

### User Role
**User messages** are requests or statements from the person using the LLM. This is where you ask questions or provide information.
```
User: How do I read a file in Python?
```

### Assistant Role
**Assistant messages** are the LLM's responses. These can be used to show examples or set a tone for future responses.
````
Assistant: Use the built-in `open()` function:
```python
with open('file.txt', 'r') as f:
    content = f.read()
```
````

In [1]:
import os
import time
import random
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import requests
from IPython.display import display, HTML
from openai import OpenAI, RateLimitError, pydantic_function_tool
from pydantic import BaseModel



## Anatomy of an LLM conversation
- We are using the newer Responses API, but the legacy prompt structuring because it is still supported and the one people have seen most.
- The biggest change here is what were called "messages" are now called "input" in the model call.

In [2]:
# Connect to an OpenAI API client
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Pick the model to use
model="gpt-4o-mini"

# Give the model a system instruction. A standard component of system prompts is to give the model an identity,
# which helps guide the output tone and content. Here we will provide a generic one, but performance on
# specific tasks may be improved if a task-specific identity is provided
system_prompt = "You are a helpful assistant."

# Ask a question, give an instruction, etc.
user_prompt = "What is NAACCR?"

# The prompts are packed in a list of JSON objects (dictionaries), with the associated role
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

# The message list is sent to our model to create a response
response = client.responses.create(
    model=model,
    input=messages
)

# You can access the output text quickly using the response's output_text attribute
print(response.output_text)

NAACCR stands for the **North American Association of Central Cancer Registries**. It is an organization that provides leadership and support to cancer registries in the United States, Canada, and Mexico. NAACCR works to improve the quality, completeness, and timeliness of cancer data collected by central cancer registries, which is crucial for cancer surveillance, research, public health planning, and policy development.

The association develops and promotes standards for cancer data collection, coding, and reporting, as well as provides education, training, and certification programs for cancer registrars and registries. NAACCR also facilitates data sharing and collaboration among registries to enhance cancer surveillance efforts across North America.


A response is actually a list of typed objects (Pydantic data models):

In [8]:
# Define a function to loop through response objects and print JSON versions
import json
def print_response_outputs(response):
    for output in response.output:
        print(json.dumps(output.model_dump(), indent=2))

# Display the previous response (in this case it's only one object). Note that the role of this response is "assistant"
print_response_outputs(response)

{
  "id": "msg_0bb40fe636356eba0069c6a4abb8288196b3b43e72f9cfd8da",
  "content": [
    {
      "annotations": [],
      "text": "NAACCR stands for the **North American Association of Central Cancer Registries**. It is an organization that promotes uniform data standards for cancer registration, provides education and training, and works to improve the quality and use of cancer data. NAACCR facilitates collaboration among cancer registries in the United States, Canada, and Mexico to collect and analyze cancer incidence and survival data, which supports research, public health planning, and cancer control efforts.",
      "type": "output_text",
      "logprobs": []
    }
  ],
  "role": "assistant",
  "status": "completed",
  "type": "message"
}


### Tools
- Built-in tools
- Define your own

In [16]:
system_prompt = "You are a helpful assistant."
user_prompt = "Search the web for the current version of the NAACCR data dictionary."
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

response = client.responses.create(
    model=model,
    input=messages,
    tools=[
        {"type": "web_search"}
    ]
)

# Now there are two output objects: one for the web search, one for the response to the prompt
print_response_outputs(response)

{
  "id": "ws_0d8b42da8eda80bf0069c6b634d734819790df3874a50fce0c",
  "action": {
    "query": "current version of NAACCR data dictionary",
    "type": "search",
    "sources": null,
    "queries": [
      "current version of NAACCR data dictionary"
    ]
  },
  "status": "searching",
  "type": "web_search_call"
}
{
  "id": "msg_0d8b42da8eda80bf0069c6b635b7a0819780cfb9e1df64bec2",
  "content": [
    {
      "annotations": [],
      "text": "The current version of the NAACCR (North American Association of Central Cancer Registries) Data Standards and Data Dictionary is Version 26, which was posted on June 23, 2025. The previous version, Version 25, was posted on May 31, 2024.\n\nYou can access the current Version 26 Data Standards and Data Dictionary directly on the NAACCR website here:  \nhttps://apps.naaccr.org/data-dictionary/data-dictionary\n\nFor more details or access to specific chapters and revisions related to Version 25, you can also visit:  \nhttps://apps.naaccr.org/data-dicti

You can also define your own tools. Let's define a function to calculate the elapsed time between two dates and then create a tool signature that we can use with the model:

In [29]:
from datetime import datetime

def elapsed_time(start_date_str: str, end_date_str: str) -> dict:
    """
    Calculate elapsed time between two dates in YYYY-MM-DD format.

    Args:
        start_date_str (str): Start date (YYYY-MM-DD)
        end_date_str (str): End date (YYYY-MM-DD)

    Returns:
        dict: elapsed time in days, weeks, and years
    """
    date_format = "%Y-%m-%d"
    
    start_date = datetime.strptime(start_date_str, date_format)
    end_date = datetime.strptime(end_date_str, date_format)
    
    delta = end_date - start_date
    
    days = delta.days
    weeks = days / 7
    years = days / 365
    
    return {
        "days": days,
        "weeks": weeks,
        "years": years
    }

elapsed_time_tool_signature = {
    "type": "function",
    "name": "elapsed_time",
    "description": "Calculate the elapsed time between two dates.",
    "parameters": {
        "type": "object",
        "properties": {
            "start_date_str": {
                "type": "string",
                "description": "A date in YYYY-MM-DD format.",
            },
            "end_date_str": {
                "type": "string",
                "description": "A date in YYYY-MM-DD-YYYY format.",
            },
        },
        "required": ["start_date_str", "end_date_str"],
    },
}


Now we can do a more complex call to the model:

In [32]:
system_prompt = "You are a helpful research assistant. Search the web for information and then run any necessary functions, if applicable"
user_prompt = "How much time elapsed between the release dates of the two most recent NAACCR data dictionary versions?"
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

response = client.responses.create(
    model=model,
    input=messages,
    tools=[
        {"type": "web_search_preview"},
        elapsed_time_tool_signature
    ]
)

# Now there are two output objects: one for the web search, one for the tool call. The arguments should match the release dates from the web search.
print_response_outputs(response)

{
  "id": "ws_046c82da64049df40069c6bc480cec8194bb6482f7fa608cc4",
  "action": {
    "query": "latest NAACCR data dictionary versions release dates",
    "type": "search",
    "sources": null,
    "queries": [
      "latest NAACCR data dictionary versions release dates"
    ]
  },
  "status": "searching",
  "type": "web_search_call"
}
{
  "arguments": "{\"start_date_str\":\"2024-05-31\",\"end_date_str\":\"2025-06-23\"}",
  "call_id": "call_exRWdriMxwxvZOR3EseYtxjX",
  "name": "elapsed_time",
  "type": "function_call",
  "id": "fc_046c82da64049df40069c6bc49082481949e598e941c371354",
  "status": "completed"
}


In [42]:
# Now we loop through the tool calls until we hit the function,
# then we run it with the arguments provided by the model and label it a "function_call_output"
for tool_call in response.output:
    if tool_call.type != "function_call":
        continue

    name = tool_call.name
    args = json.loads(tool_call.arguments)

    result = elapsed_time(**args)
    tool_outputs = [{
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result)
    }]

# We append send the original messages it back to the model
final_response = client.responses.create(
    model=model,
    input=messages + response.output + tool_outputs,
    tools=[
        {"type": "web_search_preview"},
        elapsed_time_tool_signature
    ]
)

print(final_response.output_text)

The time elapsed between the release dates of the two most recent NAACCR data dictionary versions is 388 days, which is about 1.06 years or approximately 55.4 weeks.


Note that every time you call a tool, the message list becomes longer. These messages are referred to as the **context window**. So if you are running an agent that calls tools over and over, the amount of information the model processes can become large quite rapidly. 

## Context
In LLM terminology, **context** is the text (tokens) content in a given conversation or interaction. At a minimum, this includes the system and user prompts plus the model outputs. As you can see in the tool chaining example, the outputs from each step are appended to the messages and sent back to the model. **The result is that each successive call to the model has a larger context.** If you are running an agent that is repeatedly reading files, making tool calls, etc., the context window can fill rapidly. There is a sweet-spot for context length:
- not enough context means the model has insufficient information from which to predict tokens (e.g. make decisions, output correct information)
- too much context overloads the model with information, which negatively impacts the performance of token prediction (the attention gets spread too thin)
- there is an upper limit to the context window, at which point the model will return an error
- larger context == higher costs

With those points in mind, context management becomes increasingly important for longer sessions or more complex tasks.

## Other Important Concepts

### Temperature & Sampling
- **Temperature**: Controls randomness (0 = deterministic, 1+ = more creative)
- Lower for factual tasks, higher for creative writing

### Few-Shot Prompting
Providing examples in the prompt helps the LLM understand what you want:
```
User: Convert these to snake_case:
camelCase → camel_case
PascalCase → pascal_case
myVariable → my_variable
```

### Chain of Thought
Asking the LLM to explain its reasoning can improve accuracy:
```
User: Solve this step by step...
```